# yolov8n-voc on voc — inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/voc/yolov8n-voc_infer.ipynb)

Role: inference; tier **R** on the first pass (`SUBSET = 50`), **N** on the full split (4952 images, `SUBSET = 0`). Tasks: [T35.6](../../docs/tasks/M35-yolov8n.md), [T14.11](../../docs/tasks/M14-notebooks.md);
notebooks: [M14](../../docs/tasks/M14-notebooks.md).

Prerequisites:
- data: `data/VOCdevkit/VOC2007/ImageSets/Main/test.txt` — tools/get_voc.sh
- weights: `build/notebooks/voc/yolov8n-voc/[runs/<run>/]final.npz` — notebooks/voc/yolov8n-voc_train.ipynb

Each step prints the `python tools/…` command it runs: a result can be reproduced
from the command line. Outputs in `build/notebooks/voc/yolov8n-voc/eval/`, never in `results/` (an mAP on `SUBSET`
images is not publishable, M12 rules). Notebook generated by
`python -m tools.notebooks` (`make notebooks`): edit the template
`tools/notebooks/gabarits.py`, not this file.

## Parameters

In [ ]:
NET       = 'yolov8n-voc'  # name in CFG_FILES or cfg (relative to the root)
DATASET   = 'voc'  # key of DATASETS
SPLIT     = None  # None: evaluation split of the dataset
WEIGHTS   = None  # None: final.weights of run RUN; or a path
DEVICE    = 'cpu'  # cpu | gpu (training only; the rest on CPU)
SUBSET    = 50  # images evaluated; 0: full split (tier N)
SIZE      = None  # input WxH (e.g. '640x192'); None: the one of the cfg
RESIZE    = 'v5'  # v5 (letterbox and thresholds of YOLOv5, T33.4)
PRESET    = 'v8'  # eval_voc preset with RESIZE v5: NMS 0.7 of YOLOv8 (T35.3)
METRIC    = None  # None: the one of the dataset (coco for coco and flir)
DATA_ROOT = None  # dataset root; None: data/<dataset> (Colab: Drive folder)
RUNS_DIR  = 'build/notebooks/voc/yolov8n-voc'  # runs of the _train and _sweep notebooks
RUN       = None  # None: the most recent; e.g. 'train', 'b16-sall', 'b8-s500'
COMPARE   = False  # evaluates every run and compares them (T14.11)
DRIVE_DIR = '/content/drive/MyDrive/EmbeddedCV'  # Colab: archives data/<dataset>.tar; None: no Drive
JOBS      = None  # evaluation processes; None: one per CPU core, capped by the memory (Colab TPU runtime: dozens)
SHOW      = 6  # images shown
OUT       = 'build/notebooks/voc/yolov8n-voc/eval'  # notebook outputs
REPO_URL  = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab: cloned repository
REV       = 'main'  # Colab: revision

## Environment

Local: nothing is installed or downloaded, the cell checks the prerequisites and
stops with the command to run. Colab: clone of the repository, installation, CuPy if
`DEVICE = "gpu"`, weights, then data by `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<dataset>.tar` or direct download; ExDark and FLIR: archive only,
made on the PC by `tools/get_datasets.sh pack <dataset>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone of an earlier session: reset to REV (notebook and code of the same commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules imported before the update
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # CPU runtime: no driver, CuPy would say "cudaErrorInsufficientDriver".
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("Colab runtime without a GPU: create a new Colab server "
                               "of GPU type (T4, L4 or A100), or DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # driver version
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("repository root not found: open the notebook from notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
from tools.notebooks import colab, runs

if COLAB and DRIVE_DIR:  # runs trained on Colab: copied to <DRIVE_DIR>/runs/
    env.mount_drive(DRIVE_DIR)
    colab.restore_outputs(RUNS_DIR, DRIVE_DIR)
RUNS = runs.find_runs(RUNS_DIR)
print(f"runs of {RUNS_DIR}:\n{runs.listing(RUNS)}")
CHOSEN = None
if WEIGHTS is None:
    CHOSEN = runs.pick(RUNS_DIR, RUN, hint='notebooks/voc/yolov8n-voc_train.ipynb')
    WEIGHTS, OUT = CHOSEN.weights, f"{OUT}/{CHOSEN.name}"
    print(f"chosen run: {CHOSEN.name} ({WEIGHTS})")
env.prepare(DATASET, WEIGHTS, NET, DATA_ROOT, COLAB, hint='notebooks/voc/yolov8n-voc_train.ipynb',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Network and weights

`yolo.models.tiny_yolo.build` and `yolo.io.darknet_weights.load_darknet_weights`.

In [ ]:
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import SPLIT_IMAGES, palier
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.infer.pipeline import detect, preprocess
from tools.eval_voc import load_net

net = load_net(NET, WEIGHTS)
size = input_size(net.net, parse_size(SIZE) if SIZE else None)
channels = net.net["input"][0]
family, names = datasets.model_family(DATASET, net.net["classes"])
print(f"{NET}: {net.net['classes']} classes ({family}), input {size_label(size)}, "
      f"{channels} channel(s); weights {WEIGHTS}")

## Detections on a few images

`yolo.infer.pipeline.preprocess` and `detect` (demo threshold), boxes by
`tools/detect.py:draw`.

In [ ]:
for s in datasets.load(DATASET, DATA_ROOT, SPLIT)[:SHOW]:
    with Image.open(s["image"]) as img:
        x, wh = preprocess(img, size, RESIZE, "pil", channels)
        boxes, scores, labels = detect(net, x[None], [wh], RESIZE)[0]
        shown = draw(img, boxes, scores, labels, names)
    shown.thumbnail((640, 640))
    print(f"{s['id']}: {len(labels)} detections")
    display(shown)

## Float mAP (T14.3)

`tools/eval_voc.py`, metric of the dataset (`Dataset.metric`; COCO and FLIR: AP@[.5:.95]
and AP50). PR curves of the automatic mode of M13 (VOC metric).

The chosen run is evaluated in `RUNS_DIR/eval/<run>/`, as by the comparison
below, which reuses this result: an mAP already computed for this `SUBSET` is not
computed again.

In [ ]:
print(f"{SUBSET or 'all'} images: tier "
      f"{palier(SUBSET or SPLIT_IMAGES.get(DATASET))} (M12 rules)")
if CHOSEN is not None:
    EVAL_DIR = Path(RUNS_DIR, "eval", CHOSEN.name)
    runs.evaluate(CHOSEN, NET, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                  DATA_ROOT, out_dir=EVAL_DIR, jobs=JOBS, preset=PRESET)
    map_float, dets = runs.eval_paths(EVAL_DIR, SUBSET, RESIZE)
else:
    map_float = Path(OUT) / "map_float.md"
    map_float.unlink(missing_ok=True)
    C.run(C.cmd_eval_voc(NET, WEIGHTS, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                         DATA_ROOT, out=f"{OUT}/dets", markdown=map_float, jobs=JOBS, preset=PRESET))
    dets = Path(OUT, "dets")
display(Markdown(map_float.read_text()))
for png in sorted(Path(dets, "figures").glob("pr*.png")):
    display(Image.open(png))

## Comparison of the runs (T14.11)

With `COMPARE = True`, each run found in `RUNS_DIR` (`_train` notebook and
`_sweep` sweep) is evaluated on the same `SUBSET` images; an mAP already computed
for this `SUBSET` is reused. The run evaluated above is chosen by `RUN`.

In [ ]:
if COMPARE:
    rows = [runs.evaluate(r, NET, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                          DATA_ROOT, out_dir=f"{RUNS_DIR}/eval/{r.name}", jobs=JOBS, preset=PRESET)
            for r in RUNS]
    display(Markdown(runs.table(rows)))
else:
    print("COMPARE = False: comparison skipped; runs found:")
    print(runs.listing(RUNS))

## Fine-tuned against out-of-domain

The run against the COCO weights evaluated through `MAPPINGS` by
`notebooks/voc/yolov8n-coco_infer.ipynb` (T37.2, its last execution): the
baseline the fine-tuning must beat. Compare at the same `SUBSET` (images in the
evaluation column).

In [ ]:
rows = ["| weights | mAP | evaluation |", "|---|---|---|"]
for label, table in (("fine-tuned", map_float),
                     ("COCO, out of domain", Path("build/notebooks/voc/yolov8n-coco/map_float.md"))):
    m = runs.read_map(table)
    head = (Path(table).read_text().splitlines()[0].lstrip("# ") if m is not None
            else "— (run notebooks/voc/yolov8n-coco_infer.ipynb)")
    rows.append(f"| {label} | {'—' if m is None else f'{m:.2f}'} | {head} |")
display(Markdown("\n".join(rows)))

## Summary

In [ ]:
print(map_float.read_text())
print(f"outputs: {OUT}/; revision {TRACE['rev']}")
print("equivalent commands:")
for c in C.HISTORY:
    print("  " + c)